# CAGED — estoque de empregos formais no Brasil

Este notebook é o motor de coleta e publicação da série `caged-total-mensal`.

- **Fonte:** Ministério do Trabalho e Emprego (MTE) — estatística mensal do emprego formal
  (Caged / Novo Caged). A série numérica usada aqui é a publicada pelo Banco Central no SGS,
  código **28763** — "Estoque de empregos formais — Total", unidade "Unidades", mensal.
- **Endpoint:** `https://api.bcb.gov.br/dados/serie/bcdata.sgs.28763/dados?formato=json`
- **Definição:** número de vínculos formais ativos no último dia do mês de competência
  (celetistas; inclui aprendizes, intermitentes e temporários; não inclui estatutários).
  É um **estoque** — um nível, não uma variação. Unidade `numero` (`vínculos`, 0 casas),
  `eixo: contagem`, `medida: estoque`, granularidade `mensal`.
- **Fórmula:** nenhuma. O valor é lido da série oficial, o notebook recorta os meses fechados
  e formata; o cálculo é definido pela fonte.
- **Período publicado:** de `1995-01-01` (recorte editorial, alinhado às demais séries) até o último mês fechado. A fonte tem dados desde `1992-01`, mas o recorte é mantido.
- **Arquivo gerado:** `indicadores/caged/caged-total-mensal.json`, `versao: 1`.

## Estoque ou fluxo?

**Estoque.** O Caged, na origem, é o registro de *admissões e desligamentos* (fluxo) por
competência — mas o número publicado aqui não é o saldo, é o nível de vínculos.

Três conferências contra a fonte sustentam o veredito, e a célula final do notebook as repete
a cada execução:

1. **Magnitude e persistência.** A série vale 20.838.432 em 1995-01 e 48.244.696 em 2026-08.
   Um saldo do Caged oscila em torno de zero; o maior movimento mensal de toda a série é
   **−981.331** (2020-04), 2,6% do nível — um estoque de dezenas de milhões não cai milhões
   de vínculos em um mês.
2. **Identidade contábil.** Δestoque de ago/2026 = 48.244.696 − 48.078.869 = **165.827**,
   exatamente o saldo divulgado pelo MTE para o mês (2.294.563 admissões − 2.128.736
   desligamentos). O nível se move pelo fluxo: é o fluxo acumulado, não o fluxo.
3. **Rótulo da fonte.** O metadado do SGS descreve a série como "Estoque de empregos formais
   segundo o Novo Caged", unidade "Unidades"; a apresentação oficial do MTE chama o número de
   "estoque total recuperado para o Caged".

## O nível é construído por acumulação

O metadado do SGS é explícito: "o estoque do emprego formal é calculado a partir do estoque
base do primeiro dia útil do ano corrente e evoluído a partir dos saldos mensais de admissão
líquida de demissão". Ou seja, **o estoque é derivado** — base anual declarada + fluxos
acumulados —, e não uma contagem independente de vínculos. Duas consequências práticas:

- a variação mês a mês é o saldo do mês, exceto em janeiro, quando a série recebe o novo
  estoque base e pode dar um degrau que não é fluxo;
- a cada início de ano **toda** a série é revista para o novo estoque base. Em 2026-11-01, o
  JSON publicado (vintage de out/2025) está exatamente **1.301.949** acima do vintage atual
  em todo mês de 1995-01 a 2019-12, com as variações mensais idênticas — assinatura de
  revisão de base, não de revisão de fluxo. Por isso o notebook recalcula a série inteira.

Confirmação de que a base é o nível de dezembro: `2025-12 = 47.110.663`, exatamente o
"estoque RAIS 2025" que a apresentação de ago/2026 usa como base para evoluir o ano.

## Três trechos encadeados (a emenda de 2020)

O metadado do SGS descreve a construção da série:

> "As séries divulgadas pelo Banco Central do Brasil resultam do encadeamento das séries do
> Novo Caged, disponíveis a partir de janeiro de 2020, e do Caged, no período de janeiro de
> 2007 a dezembro de 2019, ambas de acordo com a CNAE 2.0. Para o total e grandes setores as
> séries foram retroagidas, iniciando em 1992, a partir de informações do Caged disponíveis
> nessa classificação."

São, portanto, três trechos: **1992–2006** (retroação do Caged na CNAE 2.0), **2007–2019**
(Caged antigo) e **2020–** (Novo Caged, com eSocial). O encadeamento torna o nível contínuo —
2019-12 = 38.152.770 e 2020-01 = 38.267.044, +114.274 (+0,3%), sem degrau visível — mas a
*medição* muda, e o próprio BCB registra "diferenças metodológicas relevantes" entre os dois
sistemas. Leitura correta: a série é um caminho contínuo de nível; um salto em jan/2020 não
é economia, e as taxas de crescimento atravessando a emenda têm qualidade diferente das de
dentro de cada trecho.

## Execução sem sobrescrever o publicado

A variável de ambiente `CAGED_SAIDA` redireciona o arquivo de saída. Para rodar em ensaio:

```
CAGED_SAIDA=/caminho/temporario jupyter nbconvert --execute --to notebook CAGED.ipynb
```


## Coleta da série oficial

A leitura é feita desde o início da série na fonte, sem atualização incremental e sem ler o
JSON já publicado para decidir o período. A URL de origem vai também para `urlFonte`.


In [1]:
from datetime import date
from pathlib import Path
import json
import math
import os
import time

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
PASTA = RAIZ / "indicadores" / "caged" if (RAIZ / "indicadores" / "caged").is_dir() else RAIZ
DESTINO = Path(os.environ.get("CAGED_SAIDA", PASTA))
ARQUIVO = DESTINO / "caged-total-mensal.json"

SERIE = 28763
URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
URL_FONTE = URL.format(serie=SERIE) + "?formato=json"
INICIO = date(1995, 1, 1)
HOJE = date.today()


def baixar(serie, tentativas=4):
    """Baixa a série SGS inteira e devolve [(date, float)] em ordem ascendente.

    O SGS devolve 502/504 e estoura timeout de forma esporádica; a leitura é
    repetida com espera crescente antes de desistir.
    """
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(
                URL.format(serie=serie), params={"formato": "json"}, timeout=90
            )
            resposta.raise_for_status()
            break
        except (requests.HTTPError, requests.ConnectionError, requests.Timeout):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)
    bruto = resposta.json()
    if not bruto:
        raise ValueError(f"a série SGS {serie} não retornou dados")
    leituras = []
    for item in bruto:
        dia, mes, ano = (int(parte) for parte in item["data"].split("/"))
        leituras.append((date(ano, mes, dia), float(item["valor"])))
    leituras.sort()
    return leituras


def milhar(valor):
    """1234567 -> '1.234.567' (só para leitura humana no console)."""
    return f"{valor:,.0f}".replace(",", ".")


leituras = baixar(SERIE)

if len({dia for dia, _ in leituras}) != len(leituras):
    raise ValueError(f"a SGS {SERIE} retornou meses repetidos")

print(f"Série SGS {SERIE}: {len(leituras)} leituras")
print(f"Período bruto: {leituras[0][0]:%Y-%m-%d} a {leituras[-1][0]:%Y-%m-%d}")
print(f"Arquivo de saída: {ARQUIVO}")
print()
print("Primeiras 3 leituras:", [(d.isoformat(), int(v)) for d, v in leituras[:3]])
print("Últimas 3 leituras:  ", [(d.isoformat(), int(v)) for d, v in leituras[-3:]])

Série SGS 28763: 416 leituras
Período bruto: 1992-01-01 a 2026-08-01
Arquivo de saída: /Users/lucas/Workspace/personal/brasilemnumeros/dados/indicadores/caged/caged-total-mensal.json

Primeiras 3 leituras: [('1992-01-01', 21082006), ('1992-02-01', 21004453), ('1992-03-01', 20925135)]
Últimas 3 leituras:   [('2026-06-01', 48022954), ('2026-07-01', 48078869), ('2026-08-01', 48244696)]


## Recorte e diagnóstico do tipo de número

O mês corrente é descartado porque ainda está aberto (a fonte publica com ~30 dias de
defasagem). Em seguida o notebook mede o que o número **é**: compara o nível com os
movimentos mensais, o que separa estoque de fluxo sem depender de memória.


In [2]:
def indice_mensal(dia):
    """Mês absoluto: serve para exigir um ponto por mês, sem lacuna."""
    return dia.year * 12 + (dia.month - 1)


primeiro_dia_mes_corrente = HOJE.replace(day=1)
fechados = [
    (dia, valor) for dia, valor in leituras if INICIO <= dia < primeiro_dia_mes_corrente
]

if not fechados:
    raise ValueError("nenhum mês fechado no período publicado")

if fechados[0][0] != INICIO:
    raise ValueError(f"a fonte não cobre {INICIO:%Y-%m}: começa em {fechados[0][0]:%Y-%m}")

for (anterior, _), (seguinte, _) in zip(fechados, fechados[1:]):
    if indice_mensal(seguinte) - indice_mensal(anterior) != 1:
        raise ValueError(f"lacuna na série: {anterior} -> {seguinte}")

nivel = [valor for _, valor in fechados]
variacoes = [seguinte - anterior for anterior, seguinte in zip(nivel, nivel[1:])]
maior_queda = min(variacoes)
maior_alta = max(variacoes)
maior_movimento = max(abs(maior_queda), maior_alta)
maior_relativo = max(abs(v) / n for v, n in zip(variacoes, nivel[1:]))

print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print(f"Meses fechados publicados: {len(fechados)}")
print(f"Primeiro: {fechados[0][0]:%Y-%m-%d} = {milhar(nivel[0])} vínculos")
print(f"Último:   {fechados[-1][0]:%Y-%m-%d} = {milhar(nivel[-1])} vínculos")
print()
print("Teste estoque x fluxo:")
print(f"  nível mínimo da série ........ {milhar(min(nivel))}")
print(f"  maior alta mensal ............ {maior_alta:+,.0f}".replace(",", "."))
print(f"  maior queda mensal ........... {maior_queda:+,.0f}".replace(",", "."))
print(f"  maior movimento / nível ...... {100 * maior_relativo:.2f}%")
print(f"  nível / maior movimento ...... {min(nivel) / maior_movimento:.0f}x")
print()
print("Emenda Caged -> Novo Caged (jan/2020), sem degrau por construção:")
por_data = dict(fechados)
for rotulo, dia in [
    ("2019-11", date(2019, 11, 1)),
    ("2019-12", date(2019, 12, 1)),
    ("2020-01", date(2020, 1, 1)),
    ("2020-02", date(2020, 2, 1)),
]:
    print(f"  {rotulo}: {milhar(por_data[dia])}")
print(f"  variação 2019-12 -> 2020-01: {por_data[date(2020, 1, 1)] - por_data[date(2019, 12, 1)]:+,.0f}".replace(",", "."))

Mês corrente excluído: 2026-11
Meses fechados publicados: 380
Primeiro: 1995-01-01 = 20.838.432 vínculos
Último:   2026-08-01 = 48.244.696 vínculos

Teste estoque x fluxo:
  nível mínimo da série ........ 19.635.100
  maior alta mensal ............ +440.521
  maior queda mensal ........... -981.331
  maior movimento / nível ...... 2.64%
  nível / maior movimento ...... 20x

Emenda Caged -> Novo Caged (jan/2020), sem degrau por construção:
  2019-11: 38.460.081
  2019-12: 38.152.770
  2020-01: 38.267.044
  2020-02: 38.489.146
  variação 2019-12 -> 2020-01: +114.274


## Escrita do JSON do contrato

O arquivo é gravado inteiro (nada de merge com o que já está publicado), com `versao`, `id`,
`unidade`, `eixo`, `medida`, `granularidade`, `fonte`, `urlFonte` e `dados` no formato
`{"data": "YYYY-MM-DD", "valor": ...}`. O valor é inteiro — vínculo é contagem.


In [3]:
for dia, valor in fechados:
    if abs(valor - round(valor)) > 1e-6:
        raise ValueError(f"valor não inteiro em {dia}: {valor}")

pontos = [
    {"data": dia.strftime("%Y-%m-%d"), "valor": int(round(valor))}
    for dia, valor in fechados
]

serie = {
    "versao": 1,
    "id": "caged-total-mensal",
    "unidade": {"tipo": "numero", "simbolo": "vínculos", "casas": 0},
    "eixo": "contagem",
    "medida": "estoque",
    "granularidade": "mensal",
    "fonte": "Ministério do Trabalho e Emprego",
    "urlFonte": URL_FONTE,
    "dados": pontos,
}

DESTINO.mkdir(parents=True, exist_ok=True)
with ARQUIVO.open("w", encoding="utf-8") as arquivo:
    json.dump(serie, arquivo, ensure_ascii=False, indent=2)
    arquivo.write("\n")

print(f"{len(pontos)} pontos escritos em {ARQUIVO}")
print(f"Primeiro ponto: {pontos[0]}")
print(f"Último ponto:   {pontos[-1]}")

380 pontos escritos em /Users/lucas/Workspace/personal/brasilemnumeros/dados/indicadores/caged/caged-total-mensal.json
Primeiro ponto: {'data': '1995-01-01', 'valor': 20838432}
Último ponto:   {'data': '2026-08-01', 'valor': 48244696}


## Validação inline

A célula final falha se o contrato, a ordem, a unicidade e a continuidade dos meses, a
finitude dos valores, a igualdade com a fonte ou a natureza de **estoque** do número forem
violadas. Também repete as conferências contra a divulgação oficial do MTE, com tolerância
para a revisão mensal das declarações fora do prazo.


In [4]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
assert serie["versao"] == 1, serie["versao"]
assert serie["id"] == "caged-total-mensal", serie["id"]
assert serie["unidade"] == {"tipo": "numero", "simbolo": "vínculos", "casas": 0}, serie["unidade"]
assert serie["eixo"] == "contagem", serie["eixo"]
assert serie["medida"] == "estoque", serie["medida"]
assert serie["granularidade"] == "mensal", serie["granularidade"]
assert serie["fonte"] == "Ministério do Trabalho e Emprego", serie["fonte"]
assert serie["urlFonte"] == URL_FONTE, serie["urlFonte"]
assert serie["urlFonte"].startswith("https://"), serie["urlFonte"]

dados = serie["dados"]
assert dados, "série vazia"

datas = [ponto["data"] for ponto in dados]
dias = [date.fromisoformat(texto) for texto in datas]
por_data = {ponto["data"]: ponto["valor"] for ponto in dados}

for ponto in dados:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert isinstance(ponto["data"], str), ponto
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], int), ponto
    assert math.isfinite(ponto["valor"]), ponto

assert all(a < b for a, b in zip(dias, dias[1:])), "datas fora de ordem ou repetidas"
assert all(dia.day == 1 for dia in dias), "dia diferente de 01"
assert all(
    indice_mensal(b) - indice_mensal(a) == 1 for a, b in zip(dias, dias[1:])
), "mês faltando ou repetido"
assert datas[0] == "1995-01-01", datas[0]
assert dias[-1] < primeiro_dia_mes_corrente, datas[-1]
assert len(dados) == len(fechados), (len(dados), len(fechados))

# 1) O publicado é exatamente o que a fonte devolveu, sem arredondamento.
esperado = [
    {"data": dia.strftime("%Y-%m-%d"), "valor": int(round(valor))}
    for dia, valor in fechados
]
assert dados == esperado, "valor publicado difere da fonte"

# 2) O número é ESTOQUE, não fluxo. Um saldo do Caged vive perto de zero e pode
#    despencar milhões em um mês; um estoque de ~40 milhões não.
niveis = [ponto["valor"] for ponto in dados]
movimentos = [b - a for a, b in zip(niveis, niveis[1:])]
assert min(niveis) > 19_000_000, min(niveis)
assert max(abs(m) for m in movimentos) < 1_500_000, max(abs(m) for m in movimentos)
assert max(abs(m) / n for m, n in zip(movimentos, niveis[1:])) < 0.05
assert min(niveis) > 10 * max(abs(m) for m in movimentos)

# 3) Choques: quedas de nível pequenas e localizadas, como estoque deve ser.
assert por_data["2020-04-01"] - por_data["2020-03-01"] < -900_000   # pandemia
assert por_data["2008-12-01"] - por_data["2008-11-01"] < -600_000   # crise de 2008
assert por_data["2015-12-01"] - por_data["2015-11-01"] < -550_000   # recessão 2015-16
assert por_data["2020-06-01"] < por_data["2020-02-01"]              # nível ainda abaixo

# 4) Emenda de fonte: 2020-01 é contínuo com 2019-12 (encadeamento, não degrau).
salto = por_data["2020-01-01"] - por_data["2019-12-01"]
assert abs(salto) < 200_000, salto

# 5) Conferência contra a divulgação oficial do MTE (apresentação de ago/2026,
#    publicada em 2026-09-29): estoque 48.244.696 e saldo do mês 165.827
#    (2.294.563 admissões - 2.128.736 desligamentos). Tolerância de 0,15% porque
#    as declarações fora do prazo revisam os meses recentes a cada divulgação.
oficial_estoque_ago_2026 = 48_244_696
oficial_saldo_ago_2026 = 165_827
if "2026-08-01" in por_data:
    assert abs(por_data["2026-08-01"] - oficial_estoque_ago_2026) < 0.0015 * oficial_estoque_ago_2026
    delta_ago = por_data["2026-08-01"] - por_data["2026-07-01"]
    assert abs(delta_ago - oficial_saldo_ago_2026) < 5_000, delta_ago

with ARQUIVO.open(encoding="utf-8") as arquivo:
    salvo = json.load(arquivo)
assert salvo == serie, "arquivo gravado difere do objeto validado"

print(
    f"ok: {len(datas)} meses únicos, {datas[0]} a {datas[-1]}, "
    f"nível {milhar(dados[0]['valor'])} -> {milhar(dados[-1]['valor'])}, "
    f"estoque idêntico à SGS {SERIE}"
)
for rotulo in ["2008-12-01", "2015-12-01", "2020-04-01", "2021-12-01", "2026-08-01"]:
    if rotulo in por_data:
        print(f"  {rotulo}: {milhar(por_data[rotulo])}")

ok: 380 meses únicos, 1995-01-01 a 2026-08-01, nível 20.838.432 -> 48.244.696, estoque idêntico à SGS 28763
  2008-12-01: 30.849.632
  2015-12-01: 38.300.768
  2020-04-01: 37.150.202
  2021-12-01: 40.690.220
  2026-08-01: 48.244.696
